# MP1: Kontekst biznesowy i eksploracja danych

**MajsterPlus — Predykcja rezygnacji klientów**

W tym mini-projekcie:
1. Wczytasz i zweryfikujesz zbiór danych MajsterPlus
2. Zbadasz strukturę, typy danych i brakujące wartości
3. Przeanalizujesz relację między klientami a transakcjami
4. Zwizualizujesz kluczowe rozkłady i zależności
5. Wytrenujesz szybki model baseline jako „podgląd"

**Faza CRISP-DM**: Zrozumienie biznesu + Zrozumienie danych

---

## 0. Konfiguracja i reprodukowalność

In [ ]:
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Asercje wersji bibliotek
import sklearn, pandas as pd
assert sklearn.__version__.startswith(("1.4", "1.5")), (
    f"scikit-learn {sklearn.__version__} — oczekiwano 1.4.x lub 1.5.x"
)
assert pd.__version__.startswith("2."), (
    f"pandas {pd.__version__} — oczekiwano 2.x"
)
assert int(np.__version__.split(".")[0]) < 2, (
    f"numpy {np.__version__} — oczekiwano <2.0"
)
print(f"sklearn {sklearn.__version__}, pandas {pd.__version__}, numpy {np.__version__}")
print(f"Random seed: {SEED}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

## 1. Wczytywanie i weryfikacja danych

In [ ]:
import hashlib
from pathlib import Path

# Wykrywanie środowiska Colab i montowanie Google Drive
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path("/content/drive/MyDrive/PUM/2. data")
except ImportError:
    DATA_DIR = Path("../2. data")

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Nie znaleziono katalogu z danymi: {DATA_DIR}\n\n"
        "Upewnij się, że na Google Drive istnieje odpowiednia struktura folderów\n"
        "z plikami customers.csv i transactions.csv:\n\n"
        "  Mój dysk/\n"
        "    PUM/\n"
        "      2. data/\n"
        "        customers.csv\n"
        "        transactions.csv\n\n"
        "Jak to zrobić:\n"
        "  1. Wejdź na drive.google.com\n"
        "  2. Utwórz folder 'PUM' w katalogu głównym (Mój dysk)\n"
        "  3. Wewnątrz 'PUM' utwórz folder '2. data'\n"
        "  4. Wgraj oba pliki CSV z repozytorium kursu do folderu '2. data'\n\n"
        "UWAGA: Wgraj pliki bezpośrednio przez interfejs Google Drive.\n"
        "NIE otwieraj ich w Google Sheets — to zmieni formatowanie i spowoduje błąd weryfikacji."
    )

# Wczytaj i zweryfikuj customers.csv
customers = pd.read_csv(DATA_DIR / "customers.csv")
cust_md5 = hashlib.md5((DATA_DIR / "customers.csv").read_bytes()).hexdigest()
EXPECTED_CUST_MD5 = "c016cd4bfc36ac8d0a99bb6a3d55fa44"
if cust_md5 != EXPECTED_CUST_MD5:
    raise ValueError(
        f"customers.csv niezgodność MD5: {cust_md5} != {EXPECTED_CUST_MD5}\n\n"
        "Twój plik customers.csv różni się od oryginału. Najczęstsze przyczyny:\n"
        "  1. Plik został otwarty i zapisany w Google Sheets (zmienia formatowanie)\n"
        "  2. Plik został uszkodzony podczas pobierania/przesyłania\n"
        "  3. Różnice w kodowaniu końców linii (Windows vs Unix)\n\n"
        "Rozwiązanie: Pobierz ponownie oryginalny customers.csv z repozytorium kursu\n"
        "i prześlij go NA NOWO na Google Drive. NIE otwieraj go w Google Sheets —\n"
        "prześlij plik bezpośrednio przez interfejs Google Drive."
    )
print(f"customers.csv wczytano: {customers.shape}, MD5 OK")

transactions = pd.read_csv(DATA_DIR / "transactions.csv")
print(f"Wymiary transactions: {transactions.shape}")

# Wczytaj i zweryfikuj transactions.csv
txn_md5 = hashlib.md5((DATA_DIR / "transactions.csv").read_bytes()).hexdigest()
EXPECTED_TXN_MD5 = "a9c4bcfc4878baae6f5c250d4d15d737"
if txn_md5 != EXPECTED_TXN_MD5:
    raise ValueError(
        f"transactions.csv niezgodność MD5: {txn_md5} != {EXPECTED_TXN_MD5}\n\n"
        "Twój plik transactions.csv różni się od oryginału.\n"
        "Rozwiązanie: Pobierz ponownie z repozytorium kursu i prześlij na Google Drive.\n"
        "NIE otwieraj pliku w Google Sheets."
    )
print(f"transactions.csv zweryfikowano: MD5 OK")

## 2. Pierwszy rzut oka na dane

In [ ]:
# Wymiary i przegląd kolumn
print(f"Klienci: {customers.shape[0]} wierszy, {customers.shape[1]} kolumn")
print(f"\nNazwy kolumn:\n{list(customers.columns)}")

In [ ]:
# Typy danych
customers.dtypes

In [ ]:
# Pierwsze 5 wierszy
customers.head()

In [ ]:
# Podsumowanie statystyczne kolumn numerycznych
customers.describe()

## 2b. Analiza relacji między zbiorami danych

**Zadanie**: Masz dwa zbiory danych: `customers.csv` i `transactions.csv`. Zanim zaczniesz analizować poszczególne kolumny, zastanów się, jak te zbiory są ze sobą powiązane.

1. Porównaj wymiary obu zbiorów danych
2. Oblicz średnią liczbę transakcji na klienta
3. Zinterpretuj, co ten stosunek mówi o częstotliwości zakupów

*Wskazówka: Ile transakcji średnio przypada na jednego klienta? Co to mówi o częstotliwości zakupów?*

In [ ]:
# TWÓJ KOD TUTAJ
# Porównaj wymiary zbiorów customers i transactions
# Oblicz średnią liczbę transakcji na klienta
# Wypisz swoje wnioski i interpretację

## 3. Analiza brakujących wartości

**Zadanie**: Przeanalizuj brakujące wartości we wszystkich kolumnach.

1. Zidentyfikuj, które kolumny mają brakujące dane i jaki procent brakuje w każdej z nich
2. Wyświetl wyniki jako posortowaną tabelę
3. **Interpretacja**: Brak których wartości jest najbardziej niepokojący z punktu widzenia modelowania? Zastanów się: która kolumna ma najwyższy odsetek braków? Czy brak danych w tej kolumnie może być powiązany ze zmienną docelową (MAR — Missing At Random)?

In [ ]:
# TWÓJ KOD TUTAJ
# Przeanalizuj brakujące wartości — które kolumny je mają? Jaki procent brakuje?
# Wyświetl wyniki jako posortowaną tabelę (malejąco wg liczby braków)

In [ ]:
# TWÓJ KOD TUTAJ
# Interpretacja: Brak których wartości jest najbardziej niepokojący i dlaczego?
# Rozważ MAR (Missing At Random) — czy wzorzec braków może być
# powiązany z zachowaniami zakupowymi lub zmienną docelową?
# Zapisz swoją interpretację jako instrukcje print() lub komentarze

## 4. Zmienna docelowa i nierównowaga klas

In [ ]:
# Rozkład zmiennej docelowej
target_dist = customers["is_lapsed"].value_counts(normalize=True)
print("Rozkład zmiennej docelowej:")
print(target_dist)
print(f"\nWskaźnik rezygnacji: {target_dist[1]:.1%}")

fig, ax = plt.subplots(figsize=(6, 4))
customers["is_lapsed"].value_counts().plot(kind="bar", ax=ax, color=["steelblue", "coral"])
ax.set_title("Rozkład zmiennej docelowej (is_lapsed)")
ax.set_xlabel("Czy zrezygnował")
ax.set_ylabel("Liczba")
ax.set_xticklabels(["Aktywny (0)", "Zrezygnował (1)"], rotation=0)
plt.tight_layout()
plt.show()

**Zadanie**: Zastanów się nad implikacjami tego rozkładu klas.

Gdybyś zbudował(a) naiwny klasyfikator, który przewiduje WSZYSTKICH klientów jako „aktywnych" (`is_lapsed=0`), jaką dokładność (accuracy) by osiągnął? Oblicz to na podstawie powyższego rozkładu zmiennej docelowej.

*Wskazówka: Pomyśl, jaki procent danych to klasa 0...*

In [ ]:
# TWÓJ KOD TUTAJ
# Oblicz dokładność (accuracy) naiwnego klasyfikatora "przewiduj wszystkich jako aktywnych"
# Co to mówi o stosowaniu accuracy jako metryki na niezrównoważonych danych?

## 5. Mapa ciepła korelacji

**Zadanie**: Stwórz mapę ciepła korelacji wszystkich kolumn numerycznych. Jakie silne korelacje zauważasz? Które cechy mogą być problematyczne dla modelowania?

In [ ]:
# TWÓJ KOD TUTAJ
# Stwórz macierz korelacji kolumn numerycznych i zwizualizuj ją jako mapę ciepła
# Dodaj adnotacje z wartościami korelacji
# Zidentyfikuj pary z |r| > 0.5

## 6. Wykresy kluczowych zależności

**Zadanie**: Stwórz wizualizacje badające zależność między cechami a zmienną docelową `is_lapsed`. Skup się na:
- `days_since_last_purchase` w podziale na `is_lapsed`
- `purchase_count` w podziale na `is_lapsed`
- Rozkład `avg_basket_value` — czy zauważasz wartości odstające?

In [ ]:
# TWÓJ KOD TUTAJ
# Stwórz 3 wizualizacje: dwie porównujące cechy wg statusu rezygnacji
# i jedną pokazującą rozkład avg_basket_value
# Ułóż je obok siebie używając subplots

**Zadanie**: Znalazłeś/aś wartości ekstremalne w `avg_basket_value`. Zinterpretuj swoje wyniki:
- Czy to prawdopodobnie prawidłowe punkty danych, czy błędy wprowadzania danych?
- Jak mogą wpłynąć na trenowanie modelu (pomyśl o modelach liniowych vs. modele drzewiaste)?

In [ ]:
# TWÓJ KOD TUTAJ
# Zidentyfikuj wartości odstające w avg_basket_value
# Wypisz ich wartości i swoją interpretację
# Zastanów się: błąd w danych czy prawidłowa wartość? Wpływ na różne typy modeli?

## 7. Szybki podgląd — model baseline

Wytrenujmy szybko LogisticRegression na surowych cechach numerycznych, aby sprawdzić,
jak łatwo uczalny jest ten problem. Użyjemy tylko czystych kolumn numerycznych (bez preprocessingu).
To daje nam baseline do pobicia w kolejnych mini-projektach.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score

# Wybierz czyste cechy numeryczne (bez tekstu, bez kolumn z dużą liczbą braków)
# Wyklucz days_since_last_purchase — jest quasi-deterministyczna względem zmiennej docelowej
baseline_features = [
    "purchase_count", "avg_basket_value", "product_categories_bought",
    "customer_service_contacts", "store_distance_km", "account_age_days",
]

X_raw = customers[baseline_features].copy()
# Usuń wartości odstające avg_basket_value dla uczciwego baseline
mask = X_raw["avg_basket_value"] < 5000
X_raw = X_raw[mask]
y_raw = customers.loc[X_raw.index, "is_lapsed"]

X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_s, y_train)

y_pred = lr.predict(X_test_s)
y_prob = lr.predict_proba(X_test_s)[:, 1]

print(f"Baseline LogisticRegression (6 surowych cech numerycznych)")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}")
print(f"\nCzy uda się pobić ten wynik z odpowiednim preprocessingiem i feature engineering?")

## 8. Kluczowe obserwacje

**Zadanie**: Podsumuj swoje wnioski z tej eksploracji. Odpowiedz na każdy z poniższych punktów:

1. **Relacja między zbiorami danych**: Co stosunek transakcji do klientów ci powiedział?
2. **Nierównowaga klas**: Jaki jest wskaźnik rezygnacji i co oznacza dokładność naiwnego klasyfikatora dla stosowania accuracy jako metryki?
3. **Brakujące wartości**: Brak danych w której kolumnie jest najbardziej niepokojący i dlaczego?
4. **Problemy z jakością danych**: Wymień wszystkie odkryte problemy z jakością danych (pomyśl: typy, formaty, wartości odstające, niemożliwe wartości)
5. **Wynik baseline**: Jakie ROC-AUC osiągnął baseline i co to mówi o uczalności problemu?

In [ ]:
# TWÓJ KOD TUTAJ
# Zapisz swoje obserwacje jako instrukcje print, odnosząc się do każdego z 5 punktów powyżej
# Przykład:
# print("1. Relacja między zbiorami danych: ...")
# print("2. Nierównowaga klas: ...")
# print("3. Brakujące wartości: ...")
# print("4. Problemy z jakością danych: ...")
# print("5. Wynik baseline: ...")

---
*Koniec MP1 — Przejdź do MP2: Czyszczenie danych i feature engineering*